# Compare VIIRS flood data with ECMWF ecLand-CaMa-Flood

In [ ]:
import earthkit.data as ekd
import rioxarray as rxr
import xarray as xr
from affine import Affine
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.plot import show
from shapely.geometry import box
import matplotlib.pyplot as plt
import pyproj
import datetime
import os
os.environ["TMPDIR"] = os.environ.get("SCRATCH")

In [ ]:
flood_case="SriLanka"
flood_case = "Thessaly"
flood_case = "Germany"
flood_case = "Valencia"
flood_case = "Oregon"
flood_case="Pakistan"
flood_case="EmiliaRomagna"
flood_case="SE_Asia" 
flood_case="Australia"
flood_case="SouthAfrica"
flood_case="Spain"
flood_case="France"
flood_case="Persia"
flood_case="SouthItaly"
flood_case="Yangtze"

area = [5, 79, 10, 82] # SriLanka 
area = [36, 20, 41, 25] # Thessaly
area = [44, 0., 54.,10.] # Germany
area = [37, -2.5, 41,1.5] # Valencia
area = [42., -128. ,52.,-118.] # Oregon
area = [5, 60, 40, 95] #SE_Asia 
area = [46, 9, 43, 14] #EmiliaRomagna
area = [22, 66, 31, 72] # Pakistan
area = [20, 80, 30, 95] #SE_Asia 
area = [-25, 135, -10, 155]  # Australia
area = [-20, 30, -28, 38] # SouthAfrica
area = [31.5, -10, 40, 0] #Spain
area = [42, -2, 46, 2] #France
area = [29, 43, 35, 50] #Persia
area = [43, 12, 36, 19] #SouthItaly
area = [28, 105, 38, 125]  # Yangtze 

aoi = box(area[1], area[0], area[3], area[2])

output="/perm/pad/flood_cases/"+flood_case+"_flood_max.grb"
output="/perm/pad/flood_cases/"+flood_case+"_flood_mean.grb"

obs="/perm/pad/flood_cases/"+flood_case+"_flood_obs_VIIRS.tif"
msk="/perm/pad/flood_cases/"+flood_case+"_flood_mask_VIIRS.tif"

print("Benchmark case study",flood_case)
print("Data on area",area,"is saved on",output)
print("VIIRS Observation ",obs," and mask ",msk)

In [ ]:
if os.path.exists(output):
    print("File "+output+" exists!")
else:
    print("File "+output+" does not exist. Please extract and save with inundation notebook")

In [ ]:
data = ekd.from_source("file",output)
date = data.metadata("date")     # YYYYMMDD
step = data.metadata("step")     # YYYYMMDD

st=int(step[0])/24
datet=str(int(date[0])+int(st))
datet

In [ ]:
ecLand_ds = data.to_xarray()
ecLand_ds

In [ ]:
# Extract coordinates (assuming regular lat/lon)
lat = ecLand_ds.latitude.values
lon = ecLand_ds.longitude.values
grib_crs = "EPSG:4326"

# Compute grid spacing
dlat = abs(lat[1] - lat[0])
dlon = abs(lon[1] - lon[0])

# Define affine transform for coarse grid
transform = Affine.translation(lon.min() - dlon / 2, lat.min() - dlat / 2) * Affine.scale(dlon, dlat)
if transform.e > 0:
    print("grid upside down")
    transform = Affine(transform.a, transform.b, transform.c,
                           transform.d, -transform.e, transform.f)

height = len(lat)
width = len(lon)
ecLand_ds = ecLand_ds.fillna(0)
#fldfrc=np.flipud(ecLand_ds.avg_fldffr)
fldfrc=np.float32(ecLand_ds.avg_fldffr)


In [ ]:
transform

In [ ]:
gfm_hres = rxr.open_rasterio(obs)
# If it has a band dimension, squeeze it out
gfm_hres = gfm_hres.squeeze()

# Ensure binary (0/1)
gfm_hres = gfm_hres.where(gfm_hres == 1, 0)
gfm_hres = gfm_hres.fillna(0)

gfm_transform = gfm_hres.rio.transform()

if gfm_transform.e > 0:
    print("High-res raster is upside down; flipping it...")
    # Flip vertically and adjust transform
    gfm_hres.values = np.flip(gfm_hres.values, axis=-2)
    new_gfm_transform = Affine(gfm_transform.a, gfm_transform.b, gfm_transform.c,
                           gfm_transform.d, -gfm_transform.e, gfm_transform.f + (gfm_hres.shape[-2] * gfm_transform.e))
    gfm_hres.rio.write_transform(new_gfm_transform, inplace=True)

In [ ]:
gfm_msk = rxr.open_rasterio(msk)
# If it has a band dimension, squeeze it out
gfm_msk = gfm_msk.squeeze()

# Ensure binary (0/1)
gfm_msk = gfm_msk.where(gfm_msk == 1, 0)
#gfm_msk = gfm_msk.fillna(0)

gfm_msk_transform = gfm_msk.rio.transform()

if gfm_msk_transform.e > 0:
    print("High-res raster is upside down; flipping it...")
    # Flip vertically and adjust transform
    gfm_msk.values = np.flip(gfm_msk.values, axis=-2)
    new_msk_transform = Affine(gfm_msk_transform.a, gfm_msk_transform.b, gfm_msk_transform.c,
                           gfm_msk_transform.d, -gfm_msk_transform.e, gfm_msk_transform.f + (gfm_msk.shape[-2] * gfm_msk_transform.e))
    gfm_msk.rio.write_transform(new_mks_transform, inplace=True)

In [ ]:
#gfm_msk=gfm_msk*0+1

In [ ]:
#  gfm_hres

In [ ]:
# --- 3. Prepare output array (same shape as coarse grid)
fraction = np.zeros((height, width), dtype=np.float32)
missing= np.zeros((height, width), dtype=np.float32)

In [ ]:
print (gfm_msk.rio.crs)
print (gfm_transform)

In [ ]:
import numpy as np
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds

# --- Ensure source is numeric 0/1 float (no bool, no object)
src = gfm_hres.values
src = np.asarray(src)

# If xarray gives (band, y, x), take band 0
if src.ndim == 3:
    src = src[0]

src_f = src.astype(np.float32)

# --- Allocate destination correctly: MUST match dst_transform + grid dims
# You need dst_height, dst_width that correspond to your GRIB grid.
# If you already have them, set them here:
dst_height, dst_width = fraction.shape  # safest if fraction was pre-sized correctly

# If fraction wasn't allocated yet, do it like this:
# fraction = np.zeros((dst_height, dst_width), dtype=np.float32)

# Source bounds
xmin, ymin, xmax, ymax = gfm_hres.rio.bounds()

# Desired coarse grid size
dst_height, dst_width = height, width

# Build a NEW transform consistent with shape
dst_transform = from_bounds(
    xmin, ymin, xmax, ymax,
    dst_width, dst_height
)

fraction = np.zeros((dst_height, dst_width), dtype=np.float32)

reproject(
    source=gfm_hres.values.astype(np.float32),
    destination=fraction,
    src_transform=gfm_hres.rio.transform(),
    src_crs=gfm_hres.rio.crs,
    dst_transform=dst_transform,
    dst_crs=gfm_hres.rio.crs,
    resampling=Resampling.average,
)

print("Output:", np.nanmin(fraction), np.nanmax(fraction))


In [ ]:
#import numpy as np
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds

# --- Ensure source is numeric 0/1 float (no bool, no object)
src = gfm_msk.values
src = np.asarray(src)

# If xarray gives (band, y, x), take band 0
if src.ndim == 3:
    src = src[0]

src_f = src.astype(np.float32)

# --- Allocate destination correctly: MUST match dst_transform + grid dims
# You need dst_height, dst_width that correspond to your GRIB grid.
# If you already have them, set them here:
dst_height, dst_width = fraction.shape  # safest if fraction was pre-sized correctly

# If fraction wasn't allocated yet, do it like this:
# fraction = np.zeros((dst_height, dst_width), dtype=np.float32)

# Source bounds
xmin, ymin, xmax, ymax = gfm_msk.rio.bounds()

# Desired coarse grid size
dst_height, dst_width = height, width

# Build a NEW transform consistent with shape
dst_transform = from_bounds(
    xmin, ymin, xmax, ymax,
    dst_width, dst_height
)

missing = np.zeros((dst_height, dst_width), dtype=np.float32)

reproject(
    source=gfm_msk.values.astype(np.float32),
    destination=missing,
    src_transform=gfm_msk.rio.transform(),
    src_crs=gfm_hres.rio.crs,
    dst_transform=dst_transform,
    dst_crs=gfm_msk.rio.crs,
    resampling=Resampling.average,
)

print("Output:", np.nanmin(missing), np.nanmax(missing))




In [ ]:
# --- 5. Wrap result in an xarray object for convenience
missing_da = xr.DataArray(
    np.flipud(missing),
#    missing,
    dims=["lat", "lon"],
    coords={"lat": lat, "lon": lon},
    name="fraction_of_ones",
)
missing_da = missing_da.rename({"lat": "y", "lon": "x"})

# Write CRS and transform
missing_da.rio.write_crs(grib_crs, inplace=True)
missing_da.rio.write_transform(transform, inplace=True)

In [ ]:
# --- 5. Wrap result in an xarray object for convenience
fraction_da = xr.DataArray(
    np.flipud(fraction),
#    fraction,
    dims=["lat", "lon"],
    coords={"lat": lat, "lon": lon},
    name="fraction_of_ones",
)
fraction_da = fraction_da.rename({"lat": "y", "lon": "x"})

# Write CRS and transform
fraction_da.rio.write_crs(grib_crs, inplace=True)
fraction_da.rio.write_transform(transform, inplace=True)

In [ ]:
#fraction_da.rio.to_raster("/perm/pad/flood_cases/fraction_highres_on_coarse.tif")

In [ ]:
#fraction_masked = np.where(np.isnan(fldfrc),np.nan,fraction_da)

In [ ]:
diff=fldfrc-fraction_da.values

In [ ]:
mask = np.isnan(fldfrc) | (missing_da < 0.1)

fraction_masked = np.where(mask, np.nan, fraction_da)

fldfrc_masked = np.where(mask, np.nan, fldfrc)
diff_masked = np.where(mask, np.nan, diff)


In [ ]:
extent = [lon.min(), lon.max(), lat.min(), lat.max()]

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
from matplotlib.colors import LinearSegmentedColormap
g_to_n = LinearSegmentedColormap.from_list("grey_navy",["#D3D3D3", "#000080"]) # light grey → navy

# Panel 1
im1 = axes[0].imshow(fraction_masked,
                     extent=extent,
                     origin='lower',cmap=g_to_n,
                     vmin=0, vmax=1)
axes[0].set_title("VIIRS flooded area, " +datet)
axes[0].set_xlabel("Longitude(°)")
axes[0].set_ylabel("Latitude(°)")
fig.colorbar(im1, ax=axes[0]).set_label("Flooded area fraction (0.00 – 0.25)")

# Panel 2
im2 = axes[1].imshow(fldfrc_masked,
                     extent=extent,
                     origin='lower',cmap=g_to_n,
                     vmin=0, vmax=1)
axes[1].set_title("ecLand-CaMaFlood, "+datet)
axes[1].set_xlabel("Longitude(°)")
axes[1].set_ylabel("Latitude(°)")
fig.colorbar(im2, ax=axes[1]).set_label("Flooded area fraction (0.00 – 1.00)")

# Panel 3: difference
from matplotlib.colors import TwoSlopeNorm
norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

im3 = axes[2].imshow(diff_masked,
                     extent=extent,
                     origin='lower',
                     cmap="RdBu_r",
                     norm=norm)
axes[2].set_title("Difference")
axes[2].set_xlabel("Longitude(°)")
axes[2].set_ylabel("Latitude(°)")
fig.colorbar(im3, ax=axes[2]).set_label("Difference (-1.00 to 1.00)")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# --------------------------------------------------
# Plotting downsampling factor
# --------------------------------------------------
plot_stride = 4   # try 2, 4, 8 depending on memory

def to_plot_array(a, stride=1):
    """
    Lightweight plotting array:
    - convert xarray -> numpy only if needed
    - downsample for display
    - use float32 instead of float64
    """
    if hasattr(a, "values"):
        a = a.values

    a = a[::stride, ::stride]

    # avoid float64 memory
    if np.issubdtype(a.dtype, np.floating):
        a = a.astype("float32", copy=False)

    return a


fraction_plot = to_plot_array(fraction_masked, plot_stride)
fldfrc_plot   = to_plot_array(fldfrc_masked,   plot_stride)
diff_plot     = to_plot_array(diff_masked,     plot_stride)

extent = [
    float(np.nanmin(lon)),
    float(np.nanmax(lon)),
    float(np.nanmin(lat)),
    float(np.nanmax(lat)),
]

g_to_n = LinearSegmentedColormap.from_list(
    "grey_navy",
    ["#D3D3D3", "#000080"]
)

fig, axes = plt.subplots(
    1, 3,
    figsize=(14, 4),
    constrained_layout=True
)

# --------------------------------------------------
# Panel 1
# --------------------------------------------------
im1 = axes[0].imshow(
    fraction_plot,
    extent=extent,
    origin="lower",
    cmap=g_to_n,
    vmin=0,
    vmax=1,
    interpolation="nearest",
    rasterized=True
)

axes[0].set_title("VIIRS flooded area, " + datet)
axes[0].set_xlabel("Longitude (°)")
axes[0].set_ylabel("Latitude (°)")
fig.colorbar(im1, ax=axes[0]).set_label("Flooded area fraction")

# --------------------------------------------------
# Panel 2
# --------------------------------------------------
im2 = axes[1].imshow(
    fldfrc_plot,
    extent=extent,
    origin="lower",
    cmap=g_to_n,
    vmin=0,
    vmax=1,
    interpolation="nearest",
    rasterized=True
)

axes[1].set_title("ecLand-CaMaFlood, " + datet)
axes[1].set_xlabel("Longitude (°)")
axes[1].set_ylabel("Latitude (°)")
fig.colorbar(im2, ax=axes[1]).set_label("Flooded area fraction")

# --------------------------------------------------
# Panel 3: difference
# --------------------------------------------------
norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

im3 = axes[2].imshow(
    diff_plot,
    extent=extent,
    origin="lower",
    cmap="RdBu_r",
    norm=norm,
    interpolation="nearest",
    rasterized=True
)

axes[2].set_title("Difference")
axes[2].set_xlabel("Longitude (°)")
axes[2].set_ylabel("Latitude (°)")
fig.colorbar(im3, ax=axes[2]).set_label("Difference")

plt.show()

# Optional cleanup if running many plots in a loop
plt.close(fig)


In [ ]:
Flmin=0.1
gfm_flood=fraction_masked > Flmin
cmf_flood=fldfrc_masked > Flmin

In [ ]:
# compare GFM and CMF data

# compute the total number of hits, false alarms, misses and correct negatives
tp = np.where((gfm_flood == 1) & (cmf_flood == 1)  )[0].shape[0] # hits
fp = np.where((gfm_flood == 0) & (cmf_flood == 1)  )[0].shape[0] # false alarms
fn = np.where((gfm_flood == 1) & (cmf_flood == 0)  )[0].shape[0] # misses
tn = np.where((gfm_flood == 0) & (cmf_flood == 0)  )[0].shape[0] # correct negatives

csi = (tp / (tp + fp + fn))  # critical success index (CSI)
far = (fp / (tp + fp))  # false alarm ration (FAR)
hr = (tp / (tp + fn))  # hit rate (HR)

print(f'CSI: {"%.2f" % csi}')
print(f'False Alarm Ratio: {"%.2f" % far}')
print(f'Hit Rate: {"%.2f" % hr}')

# Create a grid for the evaluation results to show on a map
# create a new grid which shows whether each grid cell is a hit, false alarm or miss
eval_grid = np.zeros(gfm_flood.shape, dtype=np.uint8)
eval_grid[(gfm_flood == 1) & (cmf_flood == 1) ] = 1  # Hits
eval_grid[(gfm_flood == 1) & (cmf_flood == 0) ] = 2  # Misses
eval_grid[(gfm_flood == 0) & (cmf_flood == 1) ] = 3  # False alarms
eval_grid[(gfm_flood == 0) & (cmf_flood == 0) ] = 4  # Correct negative

eval_grid[(gfm_flood == np.nan )] = 0  # NaN
eval_grid= np.where(mask, np.nan, eval_grid)

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm

fig, axes = plt.subplots(1, 1, figsize=(8, 8), constrained_layout=True)
# Define colors for classes 1, 2, 3
colors = ["white","blue",  "red", "orange", "beige"]
cmap3 = ListedColormap(colors)

# Define boundaries: class 1, 2, 3 (values centered on integers)
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]
norm = BoundaryNorm(bounds, cmap3.N)
# Panel 1
im1 = axes.imshow(eval_grid,
                     extent=extent,norm=norm,
                     origin='lower',cmap=cmap3)
axes.set_title(flood_case+" flood evaluation: hit, miss, false-alarm, date: "+datet)
axes.set_xlabel("Longitude(°)")
axes.set_ylabel("Latitude(°)")

cbar = fig.colorbar(im1, ax=axes, boundaries=bounds, ticks=[0, 1, 2, 3, 4])

# Replace numeric ticks with text labels
cbar.ax.set_yticklabels(["no data","hit", "miss", "false-alarm", "correct-negative"])
score=f"CSI: {csi:.2f}\nFAR: {far:.2f}\nHR: {hr:.2f}"
fig.text(0.18, 0.96,score,
         fontsize=12, va='top', ha='left')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap, TwoSlopeNorm
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# ==============================================================
# COLORBLIND-SAFE COLORMAPS
# ==============================================================

# 1) Flood fractions
cmap_flood = plt.cm.Blues  # colorblind safe

# 2) Difference (diverging, colorblind safe)
cmap_diff = plt.cm.RdBu_r
norm_diff = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

# 3) Evaluation classes (Okabe–Ito palette)
eval_colors = [
    "#FFFFFF",  # No data
    "#0072B2",  # Hit
    "#D55E00",  # Miss
    "#E69F00",  # False alarm
    "#F0E442"   # Correct negative
]
cmap_eval = ListedColormap(eval_colors)
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]
norm_eval = BoundaryNorm(bounds, cmap_eval.N)

# ==============================================================
# FIGURE LAYOUT (NOW USING GEOGRAPHICAL PROJECTION)
# ==============================================================

proj = ccrs.PlateCarree()

def shift_0_360_to_m180_180(arr, lon_min, lon_max, extent):
    # extent = (lon_w, lon_e, lat_s, lat_n) in 0..360
    lon_w, lon_e, lat_s, lat_n = extent
    lon_w2 = lon_w - 360 if lon_w > 180 else lon_w
    lon_e2 = lon_e - 360 if lon_e > 180 else lon_e
    extent2 = (lon_w2, lon_e2, lat_s, lat_n)
    return extent2

extent2 = shift_0_360_to_m180_180(None, None, None, extent)

fig, axes = plt.subplots(
    2, 2, figsize=(14, 10),
    constrained_layout=True,
    subplot_kw={"projection": proj}
)

plt.rcParams["font.size"] = 11

def add_map_features(ax):
    ax.coastlines(resolution="10m", linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)

    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        color="gray",
        linestyle="--",
        linewidth=0.3
    )

    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

# ==============================================================
# PANEL A – GFM
# ==============================================================
ax = axes[0, 0]
imA = ax.imshow(
    fraction_masked,
    extent=extent2,
    origin="lower",
    cmap=cmap_flood,
    vmin=0, vmax=1,
    transform=proj
)
ax.set_title(f"A) VIIRS flooded area\nDate: {datet}")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
ax.set_extent(extent2, crs=proj)
add_map_features(ax)

cbar = fig.colorbar(imA, ax=ax, shrink=0.85)
cbar.set_label("Flooded fraction")

# ==============================================================
# PANEL B – CaMaFlood
# ==============================================================
ax = axes[0, 1]
#imB = ax.imshow(
#    fldfrc,
#    extent=extent2,
#    origin="lower",
#    cmap=cmap_flood,
#    vmin=0, vmax=1,
#    transform=proj
#)
imB = ax.imshow(
    fldfrc_masked,
    extent=extent2,
    origin="lower",
    cmap=cmap_flood,
    vmin=0, vmax=1,
    transform=proj
)
ax.set_title(f"B) ecLand–CaMaFlood\nDate: {datet}")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
ax.set_extent(extent2, crs=proj)
add_map_features(ax)

cbar = fig.colorbar(imB, ax=ax, shrink=0.85)
cbar.set_label("Flooded fraction")

# ==============================================================
# PANEL C – Difference
# ==============================================================
ax = axes[1, 0]
imC = ax.imshow(
    diff_masked,
    extent=extent2,
    origin="lower",
    cmap=cmap_diff,
    norm=norm_diff,
    transform=proj
)
ax.set_title(f"C) Difference (VIIRS – CaMaFlood)\nDate: {datet}")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
ax.set_extent(extent2, crs=proj)
add_map_features(ax)

cbar = fig.colorbar(imC, ax=ax, shrink=0.85)
cbar.set_label("Difference")

# ==============================================================
# PANEL D – Evaluation (hits/misses/etc.)
# ==============================================================
ax = axes[1, 1]
imD = ax.imshow(
    eval_grid,
    extent=extent2,
    origin="lower",
    cmap=cmap_eval,
    norm=norm_eval,
    transform=proj
)
ax.set_title(f"D) Flood Evaluation – {flood_case}\nDate: {datet}")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
ax.set_extent(extent2, crs=proj)
add_map_features(ax)

cbar = fig.colorbar(
    imD, ax=ax,
    boundaries=bounds,
    ticks=[0, 1, 2, 3, 4],
    shrink=0.85
)
cbar.ax.set_yticklabels([
    "No data", "Hit", "Miss", "False alarm", "Correct negative"
])

# Add metrics inside panel D
score_txt = f"CSI: {csi:.2f}\nFAR: {far:.2f}\nHR: {hr:.2f}"
ax.text(
    0.02, 0.98, score_txt,
    transform=ax.transAxes,
    fontsize=11, va="top", ha="left",
    bbox=dict(facecolor="white", edgecolor="gray", alpha=0.8)
)

add_map_features(ax)

plt.show()


In [ ]:
print (extent)